# 02 - Scene Description Blueprints

Now that you understand USD's fundamental building blocks, it's time to learn the essential **schemas** - the blueprints that give meaning and capabilities to your **prims**. In this module, you'll discover how to use USD's built-in schemas to create the core elements of any 3D scene.

## What You'll Learn

By the end of this module, you'll be able to:

- **Define USD schemas** - understand how IsA and API schemas define what prims are and what they can do
- **Organize scenes with Scopes and Xforms** - use these organizational containers to structure complex hierarchies
- **Simplify transforms with XformCommonAPI** - use the streamlined API for common transformation workflows
- **Apply lighting schemas** - work with the UsdLux schema domain to light your scenes

## Why This Matters

These schemas provide the standardized building blocks that ensure your content works across different applications and pipelines. A prim without a schema is just an empty container; with the right schema, it becomes a light, a transform, or a piece of geometry that other applications can understand and work with.

In [1]:
import sys
print("Python:", sys.version.split()[0])

from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade, UsdLux
print("pxr import OK.")

from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9
pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/02_scene_description_blueprints/_assets


In [2]:
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


## 2.1  Schemas

Schemas are the blueprints that give prims their meaning. In OpenUSD there are two flavors: **IsA schemas** (also called typed schemas) declare *what a prim is* via its `typeName`, while **API schemas** *add capabilities* to a prim that already has a type. A prim can only be one IsA schema at a time, but it may carry any number of API schemas listed in its `apiSchemas` metadata. IsA schemas derive from `UsdTyped` and are instantiated with `.Define(stage, path)`. Applied API schemas are attached with `.Apply(prim)` and queried with `prim.HasAPI(...)`. Together they form a flexible, extensible system for modeling everything from geometry to physics.

**Key APIs**
- `UsdGeom.Sphere.Define` - typed schema instantiation
- `UsdShade.MaterialBindingAPI.Apply` - applied API schema
- `prim.IsA(...)` / `prim.HasAPI(...)` - schema queries
- `Usd.SchemaRegistry.FindSchemaInfo`

Source: [../docs/scene-description-blueprints/schemas.md](../docs/scene-description-blueprints/schemas.md)

The source lesson contains no executable `code-cell` blocks, so we add a defensive demo below that exercises both schema types.

In [3]:
from pxr import Usd, UsdGeom, UsdShade

file_path = "_assets/schemas_demo.usda"
stage = create_new_stage(file_path)

# IsA (typed) schema: defines WHAT the prim is
sphere = UsdGeom.Sphere.Define(stage, "/World/Sphere")
sphere.GetRadiusAttr().Set(10.0)

# API schema: adds capability to an already-typed prim
binding_api = UsdShade.MaterialBindingAPI.Apply(sphere.GetPrim())

# Introspection
prim = sphere.GetPrim()
print("typeName:", prim.GetTypeName())
print("IsA Sphere:", prim.IsA(UsdGeom.Sphere))
print("HasAPI MaterialBindingAPI:", prim.HasAPI(UsdShade.MaterialBindingAPI))
print("applied API schemas:", prim.GetAppliedSchemas())

stage.Save()

typeName: Sphere
IsA Sphere: True
HasAPI MaterialBindingAPI: True
applied API schemas: ['MaterialBindingAPI']


In [4]:
DisplayUSD(file_path, show_usd_code=True)

**What just happened:** We created a `UsdGeom.Sphere` (an IsA schema that sets `typeName = "Sphere"`) and then layered a `UsdShade.MaterialBindingAPI` onto the same prim. The `apiSchemas` metadata now lists `MaterialBindingAPI`, but the prim is still fundamentally a Sphere. This shows the IsA-vs-API distinction in one prim.

## 2.2  Scope

A `Scope` is a non-renderable prim used purely for **organization**. Think of it as a folder in the scenegraph: it groups related prims without introducing any transform of its own. Because Scopes are not transformable, they are lightweight and make excellent partitions for large asset libraries, actor/environment groupings, or materials. A powerful pattern is to deactivate an entire Scope to hide its subtree from composition and rendering.

**Key APIs**
- `UsdGeom.Scope.Define(stage, path)`
- `prim.IsA(UsdGeom.Scope)`
- `prim.SetActive(False)` - deactivate a subtree

Source: [../docs/scene-description-blueprints/scope.md](../docs/scene-description-blueprints/scope.md)

In [5]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

In [6]:
from pxr import Usd, UsdGeom, Gf

file_path = "_assets/scope.usda"
stage = create_new_stage(file_path)

# World container (transformable)
world = UsdGeom.Xform.Define(stage, "/World")

num_a_prims = 2
num_b_prims = 2

# Two organizational Scopes (non-transformable grouping prims)
a_scope = UsdGeom.Scope.Define(stage, world.GetPath().AppendPath("A_Scope"))
b_scope = UsdGeom.Scope.Define(stage, world.GetPath().AppendPath("B_Scope"))

# Populate the scopes with some geometry
for a in range(num_a_prims):
    sphere = UsdGeom.Sphere.Define(stage, a_scope.GetPath().AppendPath(f"A_Sphere_{a}"))
    UsdGeom.XformCommonAPI(sphere).SetTranslate(Gf.Vec3d(a*2.5, 0, 0))

for b in range(num_b_prims):
    cube = UsdGeom.Cube.Define(stage, b_scope.GetPath().AppendPath(f"B_Cube_{b}"))
    UsdGeom.XformCommonAPI(cube).SetTranslate(Gf.Vec3d(b*2.5, -2.5, 0))

# Deactivate the A_Scope
a_scope.GetPrim().SetActive(False)

stage.Save()

In [7]:
DisplayUSD(file_path, show_usd_code=True)

**What just happened:** We built a `/World` Xform holding two Scopes, `A_Scope` and `B_Scope`, each containing placed primitives. Although we deactivated `A_Scope`, the scene description is still authored on the layer - the prim simply won't contribute to composition until reactivated. Notice that Scopes themselves carry no `xformOp` attributes.

## 2.3  Xform

Where Scope is a pure folder, `Xform` is a folder *with a transform*. An Xform prim stores translation, rotation, and scale data that cascades down to every descendant. Xforms are the backbone of scene hierarchy - you rig characters, layout environments, and animate assets by nesting geometry under Xforms and moving the parents. The underlying machinery is `UsdGeomXformable`, which exposes an ordered stack of `xformOp` attributes through `GetXformOpOrderAttr()` and `AddXformOp(...)`.

**Key APIs**
- `UsdGeom.Xform.Define(stage, path)`
- `UsdGeom.Xformable(prim).GetXformOpOrderAttr()`
- `xform.AddXformOp(opType, precision)`

Source: [../docs/scene-description-blueprints/xform.md](../docs/scene-description-blueprints/xform.md)

In [8]:
from lousd.utils.helperfunctions import create_new_stage

In [9]:
# Import the necessary modules from the pxr package:
from pxr import Usd, UsdGeom

# Create a new USD stage with root layer named "xform_prim.usda":
file_path = "_assets/xform_prim.usda"
stage: Usd.Stage = create_new_stage(file_path)

# Define a new Xform primitive at the path "/World" on the current stage:
world: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

# Save changes to the current stage to its root layer:
stage.Save()
print(stage.ExportToString(addSourceFileComment=False))

#usda 1.0

def Xform "World"
{
}




In [10]:
DisplayUSD(file_path, show_usd_code=True)

**What just happened:** We created the minimal Xform scene: a single `/World` prim of type `Xform`. No `xformOpOrder` has been authored yet, but the prim is ready to accept transform ops via `XformCommonAPI` or direct `AddXformOp` calls.

## 2.4  XformCommonAPI

`XformCommonAPI` is a non-applied API schema that gives you a sane, interchange-friendly way to author translate / rotate / scale / pivot on any `Xformable` prim. Under the hood USD allows arbitrary stacks of `xformOp` attributes, which is powerful but tricky for interchange - `XformCommonAPI` enforces a canonical op order so tools can round-trip the values. Use it for the common case; reach for raw `AddXformOp` only when you need exotic stacks.

**Key APIs**
- `UsdGeom.XformCommonAPI(prim)` - bool-convertible compatibility check
- `SetTranslate`, `SetRotate`, `SetScale`, `SetPivot`
- `GetXformOpOrderAttr()` to inspect the authored op stack

Source: [../docs/scene-description-blueprints/xformcommonapi.md](../docs/scene-description-blueprints/xformcommonapi.md)

In [11]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

In [12]:
from pxr import Usd, UsdGeom, Gf

file_path = "_assets/xformcommonapi.usda"
stage = create_new_stage(file_path)

# A root transform group we will move and rotate
world = UsdGeom.Xform.Define(stage, "/World")
parent = UsdGeom.Xform.Define(stage, world.GetPath().AppendPath("Parent_Prim"))

# Parent Translate, Rotate, Scale using XformCommonAPI
parent_xform_api = UsdGeom.XformCommonAPI(parent)
parent_xform_api.SetTranslate(Gf.Vec3d(5, 0, 3))
parent_xform_api.SetRotate(Gf.Vec3f(90, 0, 0))
parent_xform_api.SetScale(Gf.Vec3f(3.0, 3.0, 3.0))


child_translation = Gf.Vec3d(2, 0, 0)

# Child A - inherits parent transforms
child_a_cone = UsdGeom.Cone.Define(stage, parent.GetPath().AppendChild("Child_A"))
child_a_xform_api = UsdGeom.XformCommonAPI(child_a_cone)
child_a_xform_api.SetTranslate(child_translation)  # Parent_Prim transform + local placement

# Child B - "/World/Alt_Parent/Child_B" does NOT inherit Parent_Prim transforms
alt_parent = UsdGeom.Xform.Define(stage, world.GetPath().AppendChild("Alt_Parent"))
child_b_cone = UsdGeom.Cone.Define(stage, alt_parent.GetPath().AppendChild("Child_B"))
child_b_xform_api = UsdGeom.XformCommonAPI(child_b_cone)
child_b_xform_api.SetTranslate(child_translation)  # local placement only

# Inspect the authored Xform Operation Order
print("Parent xformOpOrder:", UsdGeom.Xformable(parent).GetXformOpOrderAttr().Get())
print("Alt_Parent xformOpOrder:", UsdGeom.Xformable(alt_parent).GetXformOpOrderAttr().Get())
print("Child A xformOpOrder:", UsdGeom.Xformable(child_a_cone).GetXformOpOrderAttr().Get())
print("Child B xformOpOrder:", UsdGeom.Xformable(child_b_cone).GetXformOpOrderAttr().Get())

stage.Save()

Parent xformOpOrder: [xformOp:translate, xformOp:rotateXYZ, xformOp:scale]
Alt_Parent xformOpOrder: None
Child A xformOpOrder: [xformOp:translate]
Child B xformOpOrder: [xformOp:translate]


In [13]:
DisplayUSD(file_path, show_usd_code=True)

**What just happened:** `Parent_Prim` received the canonical `xformOp:translate`, `xformOp:rotateXYZ`, `xformOp:scale` op stack via `XformCommonAPI`. `Child_A` inherits those because it sits under `Parent_Prim`, while `Child_B`, sibling-parented under `Alt_Parent`, only has its own local translate. The printed `xformOpOrder` attributes confirm the canonical ordering.

## 2.5  Lights

`UsdLux` is the standardized light schema domain. It provides typed schemas for every common physical light - `DistantLight`, `SphereLight`, `DiskLight`, `RectLight`, `CylinderLight`, `DomeLight`, `PortalLight` - plus `LightAPI` which supplies shared attributes like intensity, color, and exposure. Because lights are `Xformable`, you position them with the same `XformCommonAPI` you use for geometry. Using standard UsdLux schemas ensures renderers and DCCs interpret your lights consistently.

**Key APIs**
- `UsdLux.DistantLight.Define`, `UsdLux.SphereLight.Define`
- `light.GetIntensityAttr()`, `light.GetColorAttr()`
- `UsdGeom.XformCommonAPI(light)` to place/orient lights

Source: [../docs/scene-description-blueprints/lights.md](../docs/scene-description-blueprints/lights.md)

In [14]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

In [15]:
from pxr import Usd, UsdGeom, UsdLux

file_path = "_assets/distant_light.usda"
stage: Usd.Stage = create_new_stage(file_path)

world: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")
geo_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, world.GetPath().AppendPath("Geometry"))
box_geo: UsdGeom.Cube = UsdGeom.Cube.Define(stage, geo_scope.GetPath().AppendPath("Cube"))

# Define a new Scope primitive at the path "/World/Lights" on the current stage:
lights_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, world.GetPath().AppendPath("Lights"))

# Define a new DistantLight primitive at the path "/World/Lights/SunLight" on the current stage:
distant_light: UsdLux.DistantLight = UsdLux.DistantLight.Define(stage, lights_scope.GetPath().AppendPath("SunLight"))

stage.Save()

In [16]:
DisplayUSD(file_path, show_usd_code=True)

**What just happened:** We established a typical scene layout - `/World` Xform, a `Geometry` Scope with a Cube, and a parallel `Lights` Scope containing a `DistantLight` called `SunLight`. DistantLights emit along their local -Z axis, making them ideal for sunlight. The light won't render visibly here but now exists in the hierarchy for any Hydra renderer.

In [17]:
from pxr import Gf, Usd, UsdGeom, UsdLux

file_path = "_assets/light_props.usda"
stage: Usd.Stage = create_new_stage(file_path)
geom_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, "/Geometry")
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, geom_scope.GetPath().AppendPath("Box"))

# Define a `Scope` Prim in stage at `/Lights`:
lights_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, "/Lights")

# Define a `Sun` prim in stage as a child of `lights_scope`, called `Sun`:
distant_light = UsdLux.DistantLight.Define(stage, lights_scope.GetPath().AppendPath("Sun"))
# Define a `SphereLight` prim in stage as a child of lights_scope called `SphereLight`:
sphere_light = UsdLux.SphereLight.Define(stage, lights_scope.GetPath().AppendPath("SphereLight"))

# Configure the distant light's emissive attributes:
distant_light.GetColorAttr().Set(Gf.Vec3f(1.0, 0.0, 0.0)) # Light color (red)
distant_light.GetIntensityAttr().Set(120.0) # Light intensity
# Lights are Xformable
if not (distant_light_xform_api := UsdGeom.XformCommonAPI(distant_light)):
    raise Exception("Prim not compatible with XformCommonAPI")
distant_light_xform_api.SetRotate((45.0, 0.0, 0.0))


# Configure the sphere light's emissive attributes:
sphere_light.GetColorAttr().Set(Gf.Vec3f(0.0, 0.0, 1.0)) # Light color (blue)
sphere_light.GetIntensityAttr().Set(50000.0) # Light intensity
# Lights are Xformable
if not (sphere_light_xform_api := UsdGeom.XformCommonAPI(sphere_light)):
    raise Exception("Prim not compatible with XformCommonAPI")
sphere_light_xform_api.SetTranslate((5.0, 10.0, 0.0))

stage.Save()

In [18]:
DisplayUSD(file_path, show_usd_code=True)

**What just happened:** Two different light types now live under `/Lights`: a red `DistantLight` rotated 45 degrees on X (sunrise-ish), and a blue `SphereLight` translated up and to the right with an intensity of 50000. Because both lights are Xformable, `XformCommonAPI` works on them exactly like it does for geometry.

## Key Takeaways

- **Schemas** give prims meaning. IsA schemas declare *what* a prim is; API schemas attach additional *capabilities*.
- **Scope** is a non-transformable folder - perfect for logical grouping and bulk activation/deactivation.
- **Xform** is a transformable folder. Transforms cascade down the hierarchy to every descendant.
- **XformCommonAPI** is the interchange-safe way to author translate / rotate / scale / pivot. Prefer it over raw xformOp stacks.
- **UsdLux** provides standard light types (Distant, Sphere, Disk, Rect, Dome...) that are also Xformable, so placing lights uses the same patterns as placing geometry.

## Next up -> 03_composition_basics.ipynb

With prims, schemas, Scopes, Xforms, and lights under your belt, you're ready to learn how USD **composes** layers together into a single scenegraph. The next notebook dives into sublayers, references, payloads, variants, and inherits - the tools that let teams collaborate non-destructively on the same scene.